
# Rasters con valores bioclimáticos y precipitación anual

## bio *Bioclimáticos*
## prec *Precipitación*
# _________________________________________________________________________________________________________________

**Caragando librería e importando paquetes para uso en el código**


In [2]:
# Librerías y paquetes necesarios para poder operar los rasters.
!pip install geopandas rasterio pandas tqdm

import geopandas as gpd
import geopandas as gpf
import pandas as pd
import rasterio
import glob
import os
from tqdm import tqdm

# *********************************************************************************************************************
# DIRECTORIOS 
## DEFINIENDO DIRECTORIOS
Se definen las rutas para la obtención de los datos (archivos .tiff) que contienen tod la información climática desde el año 1970 hasta el añ0 2000. 
# *********************************************************************************************************************

In [3]:
# Verificando si la carpeta existe

root_dir = "/home/guillermo/capas/"
if os.path.exists(root_dir) and os.path.isdir(root_dir): #Verificación si es una carpeta y no un archivo.
    print(f"La carpeta {root_dir} existe.")
    
    # Buscar todos los .tif en subcarpetas
    archivos = []
    for carpeta, subcarpetas, files in os.walk(root_dir):
        for f in files:
            if f.endswith(".tif"):
                archivos.append(os.path.join(carpeta, f))
    
    if archivos:
        print(f"Se encontraron {len(archivos)} archivos .tif en {root_dir} (incluyendo subcarpetas).")
    else:
        print(f"No se encontraron archivos .tif en {root_dir}.")
else:
    print(f"La carpeta {root_dir} no existe.")

La carpeta /home/guillermo/capas/ existe.
Se encontraron 232 archivos .tif en /home/guillermo/capas/ (incluyendo subcarpetas).


# *********************************************************************************************************************
## Listando los archivos.
Se crean una lista con los archivos **.tiff** y a su vez se crea la metadata del diccionario a partir de los acrónimos
# *********************************************************************************************************************

In [4]:
# Listando los archivos .tif en subcarpetas
raster_files = glob.glob(root_dir + "/**/*.tif", recursive=True)

# Creando los acrónimos automáticos (nombre del archivo sin extensión) para crear el diccionario
acronyms = [f.split("\\")[-1].replace(".tif", "") for f in raster_files]  # Windows path
# Si usas Linux/Mac cambia a: f.split("/")[-1]


# *********************************************************************************************************************
## Obteniendo la información de las accesiones.
Se obtiene la información de las accesiones que han sido previamente extraida y transformada
# *********************************************************************************************************************

In [5]:
# Cargargando accesiones que ya fueron limpiadas (ejemplo CSV con lon y lat)
file_path = "/home/guillermo/Accesiones_Phaseolus_vulgaris/Phaseolus_vulgaris_CLEAN.csv"

# Verificar si el archivo existe antes de cargarlo
if os.path.exists(file_path):
    try:
        accesiones = pd.read_csv(file_path)
        print(f"El archivo fue cargado correctamente: {file_path}")
        print(f"El dataset tiene {accesiones.shape[0]} filas y {accesiones.shape[1]} columnas.")
    except Exception as e:
        print(f"Error al cargar el archivo: {e}")
else:
    print(f"El archivo no existe en la ruta: {file_path}")

El archivo fue cargado correctamente: /home/guillermo/Accesiones_Phaseolus_vulgaris/Phaseolus_vulgaris_CLEAN.csv
El dataset tiene 2768 filas y 51 columnas.


# *********************************************************************************************************************
## Convirtiendo a GeoDataframe.
En este paso lo que hago es transformar el dataframe convencional de pandas (accesiones) en un GeoDataFrame de geopandas (gdf). Esto permite generar una estructura espacial para poder de manejar coordenadas geográficas. En otras palabras, lo que se hace aquí es transformar la tabla de datos tabulares en una capa geográfica donde cada registro se representa como un punto en el espacio (latitud y longitud).
Pandas maneja números y textos mientras Geopandas maneja geometrías. Esto me permite representar cada accesión como un punto geográfico dentro del sistema de referencia WGS84 (EPSG:4326), compatible con las capas raster de WorldClim y otras.
De esta forma, puedo realizar análisis espaciales, como la extracción de variables climáticas o la visualización de patrones ecológicos.
# *********************************************************************************************************************


In [6]:
# Convirtiendo las accesiones accesiones a GeoDataFrame

gdf = gpd.GeoDataFrame(
    accesiones,
    geometry=gpd.points_from_xy(accesiones.DECLONGITUDE, accesiones.DECLATITUDE), #Esta parte crea una columna de geometría usando las coordenadas x (longitud) y y (latitud).
    #crs significa Coordinate Reference System (Sistema de Referencia de Coordenadas). EPSG:4326 corresponde al sistema WGS84, usado por GPS y por la mayoría de capas globales (incluido WorldClim).
    crs="EPSG:4326" 
)

# *********************************************************************************************************************
## Extrayendo los valores de las capas rasters.
En este punto extraigo los valores de la capa rasters en los puntos de ubicación de las accesiones.
Por cada accesión (punto), busco en qué celda del raster cae y obtengo el valor de esa celda.
Esto constituye el enlace entre los datos genéticos/geográficos y los datos ambientales lo que permite construir una base enriquecida con variables climáticas que caracterizan el ambiente de origen de cada accesión.
# *********************************************************************************************************************


In [7]:
# Extrayendo los valores de los rasters en puntos
def extraer_raster(rfile, gdf):
    #abre el archivo raster en modo lectura. El prefijo with asegura que el archivo se cierre automáticamente al finalizar.src es el objeto raster (una matriz de valores distribuidos en el espacio).
    with rasterio.open(rfile) as src:
        valores = [x[0] for x in src.sample(zip(gdf.geometry.x, gdf.geometry.y))]
    return valores

# *********************************************************************************************************************
## Extrayendo los valores de los rasters
Este bloque de código recorre automáticamente todas las capas raster (bioclimáticas o mensuales) para extraer sus valores climáticos en las coordenadas de las accesiones.
Luego, va almacenando los resultados en un DataFrame estructurado, donde cada columna representa una variable ambiental (por ejemplo, bio1, bio2, prec1, etc.), y cada fila corresponde a una accesión.
Aquí es donde se crea la matriz ambiental completa de todas las accesiones y variables.
# *********************************************************************************************************************


In [8]:
# Extrayendo los valores de todos los rasters 
valores = pd.DataFrame()

for i, rfile in enumerate(tqdm(raster_files, desc="Extrayendo rasters")):
    try:
        valores[acronyms[i]] = extraer_raster(rfile, gdf)
    except Exception as e:
        print(f"Error en {rfile}: {e}")
        valores[acronyms[i]] = [None] * len(gdf)


Extrayendo rasters:  43%|████████████████████████████████████████████████████████████████▋                                                                                     | 100/232 [02:02<01:55,  1.14it/s]/tmp/ipykernel_4442/1106133152.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  valores[acronyms[i]] = extraer_raster(rfile, gdf)
Extrayendo rasters:  44%|█████████████████████████████████████████████████████████████████▎                                                                                    | 101/232 [02:03<01:51,  1.18it/s]/tmp/ipykernel_4442/1106133152.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at 

In [9]:
valores.shape


(2768, 232)

# Uniendo la información
Una vez que he obtenido toda la información lo que hago es unir las dos fuentes en un Dataframe. Hago una integración de los datos (Accesiones <-> datos climáticos)
# *********************************************************************************************************************


In [10]:
# Uniendo las accesiones con los valores climáticos valores climáticos
resultado = pd.concat([accesiones, valores], axis=1)

# PENDIENTE POR DOCUMENTAR

In [11]:
import pandas as pd

# -----------------------------
# BIOCLIM 1–19
# -----------------------------
bio_metadata = pd.DataFrame({
    "Acronym": [f"bio{i}" for i in range(1,20)],
    "Unidad": ["°C"]*11 + ["mm"]*8,
    "Descripcion": [
        "Temperatura media anual",
        "Rango medio diurno (Tmax–Tmin)",
        "Isotermalidad (bio2/bio7 ×100)",
        "Estacionalidad de la temperatura (SD ×100)",
        "Temperatura máxima del mes más cálido",
        "Temperatura mínima del mes más frío",
        "Rango anual de temperatura (bio5–bio6)",
        "Temperatura media del trimestre más húmedo",
        "Temperatura media del trimestre más seco",
        "Temperatura media del trimestre más cálido",
        "Temperatura media del trimestre más frío",
        "Precipitación anual",
        "Precipitación del mes más húmedo",
        "Precipitación del mes más seco",
        "Estacionalidad de la precipitación (CV)",
        "Precipitación del trimestre más húmedo",
        "Precipitación del trimestre más seco",
        "Precipitación del trimestre más cálido",
        "Precipitación del trimestre más frío"
    ]
})

# -----------------------------
# VARIABLES MENSUALES
# -----------------------------
meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio",
         "Julio","Agosto","Septiembre","Octubre","Noviembre","Diciembre"]

tmin_metadata = pd.DataFrame({
    "Acronym": [f"tmin_{i}" for i in range(1,13)],
    "Unidad": ["°C"]*12,
    "Descripcion": [f"Temperatura mínima de {m}" for m in meses]
})

tmax_metadata = pd.DataFrame({
    "Acronym": [f"tmax_{i}" for i in range(1,13)],
    "Unidad": ["°C"]*12,
    "Descripcion": [f"Temperatura máxima de {m}" for m in meses]
})

tavg_metadata = pd.DataFrame({
    "Acronym": [f"tavg_{i}" for i in range(1,13)],
    "Unidad": ["°C"]*12,
    "Descripcion": [f"Temperatura promedio de {m}" for m in meses]
})

prec_metadata = pd.DataFrame({
    "Acronym": [f"prec_{i}" for i in range(1,13)],
    "Unidad": ["mm"]*12,
    "Descripcion": [f"Precipitación de {m}" for m in meses]
})

# -----------------------------
# UNIR TODO
# -----------------------------
metadata_extended = pd.concat([
    bio_metadata,
    tmin_metadata,
    tmax_metadata,
    tavg_metadata,
    prec_metadata
], ignore_index=True)

# Guardar a CSV para tener documentación oficial
metadata_extended.to_excel("/home/guillermo/capas/worldclim/metadata_extended.xlsx", index=False)

print(f"Metadata extendida guardada en /home/guillermo/capas/worldclim/metadata_extended.csv")
print(metadata_extended.head(10))


Metadata extendida guardada en /home/guillermo/capas/worldclim/metadata_extended.csv
  Acronym Unidad                                 Descripcion
0    bio1     °C                     Temperatura media anual
1    bio2     °C              Rango medio diurno (Tmax–Tmin)
2    bio3     °C              Isotermalidad (bio2/bio7 ×100)
3    bio4     °C  Estacionalidad de la temperatura (SD ×100)
4    bio5     °C       Temperatura máxima del mes más cálido
5    bio6     °C         Temperatura mínima del mes más frío
6    bio7     °C      Rango anual de temperatura (bio5–bio6)
7    bio8     °C  Temperatura media del trimestre más húmedo
8    bio9     °C    Temperatura media del trimestre más seco
9   bio10     °C  Temperatura media del trimestre más cálido


In [14]:
# Generando la tabla de los meses con unidades y descripción para las precipitaciones
def make_monthly(prefix, unidad, desc):
    return pd.DataFrame({
        "Acronym": [f"{prefix}{i}" for i in range(1,13)],
        "Unidad": unidad,
        "Descripcion": [f"{desc} de {m}" for m in [
            "enero","febrero","marzo","abril","mayo","junio",
            "julio","agosto","septiembre","octubre","noviembre","diciembre"
        ]]
    })

prec  = make_monthly("prec", "mm", "Precipitación")
srad  = make_monthly("srad", "kJ m-2 día-1", "Radiación solar")
tavg  = make_monthly("tavg", "°C", "Temperatura media")
tmax  = make_monthly("tmax", "°C", "Temperatura máxima")
tmin  = make_monthly("tmin", "°C", "Temperatura mínima")
vapr  = make_monthly("vapr", "kPa", "Presión de vapor")
wind  = make_monthly("wind", "m/s", "Velocidad del viento")

metadata_oficial = pd.concat([bio_metadata, prec],ignore_index=True)#, srad, tavg, tmax, tmin, vapr, wind], ignore_index=True)

In [15]:
# Creando la metadata final enlazando con archivos cargados
metadata_final = pd.DataFrame({
    "Acronym": acronyms,
    "Variable": [f.split("\\")[-1] for f in raster_files],  # si es Linux usar split("/")
    "Path": raster_files
}).merge(metadata_oficial, on="Acronym", how="left")

In [16]:

# Guardando los resultados de la metadata y de las accesiones con variables climáticas resultados
resultado.to_excel("/home/guillermo/Accesiones_Phaseolus_vulgaris/accesiones_con_worldclim.xlsx", index=False)
metadata_final.to_excel("/home/guillermo/Accesiones_Phaseolus_vulgaris/metadata_worldclim.xlsx", index=False)
#resultado.to_csv("/home/guillermo/Accesiones_Phaseolus_vulgaris/accesiones_con_worldclim.csv", index=False, encoding="utf-8")
#metadata_final.to_csv("/home/guillermo/Accesiones_Phaseolus_vulgaris/metadata_worldclim.csv", index=False, encoding="utf-8")

print("✅ Listo: accesiones + variables climáticas guardadas, con metadata completa.")


✅ Listo: accesiones + variables climáticas guardadas, con metadata completa.


In [17]:
resultado.head()

,INSTCODE,DOI,ACCENUMB,HISTORIC,CURATION,GENUS,SPECIES,SPAUTHOR,SUBTAXA,SUBTAUTHOR,...,/home/guillermo/capas/worldclim/vapr/vapr_5,/home/guillermo/capas/worldclim/vapr/vapr_3,/home/guillermo/capas/worldclim/vapr/vapr_10,/home/guillermo/capas/worldclim/vapr/vapr_2,/home/guillermo/capas/worldclim/vapr/vapr_7,/home/guillermo/capas/worldclim/vapr/vapr_1,/home/guillermo/capas/envirem/envirem/current_30arcsec_continentality,/home/guillermo/capas/envirem/envirem/current_30arcsec_climaticMoistureIndex,/home/guillermo/capas/envirem/envirem/current_30arcsec_annualPET,/home/guillermo/capas/envirem/envirem/current_30arcsec_aridityIndexThornthwaite
0,COL003,10.18730/JKECH,G16570,False,FULL,Phaseolus,vulgaris,NaN,NaN,NaN,...,0.93,0.59,0.82,0.49,1.20,0.47,-9999.0,-9999.0,-9999.0,-9999.0
1,COL003,10.18730/JKEKR,G16576,False,FULL,Phaseolus,vulgaris,NaN,NaN,NaN,...,0.81,0.59,0.65,0.51,0.78,0.47,-9999.0,-9999.0,-9999.0,-9999.0
2,COL003,10.18730/JZXX0,G990,False,FULL,Phaseolus,vulgaris,NaN,NaN,NaN,...,1.13,1.03,1.21,0.98,1.38,0.97,-9999.0,-9999.0,-9999.0,-9999.0
3,COL003,10.18730/JZXWU,G989,False,FULL,Phaseolus,vulgaris,NaN,NaN,NaN,...,1.00,0.84,1.00,0.76,1.33,0.76,-9999.0,-9999.0,-9999.0,-9999.0
4,COL003,10.18730/JZXT$,G987,False,FULL,Phaseolus,vulgaris,NaN,NaN,NaN,...,0.73,0.62,0.64,0.55,0.90,0.53,-9999.0,-9999.0,-9999.0,-9999.0
